# Data Cleaning with Python

Before you start, make sure that you are familiar with the basic usage of Jupyter Notebook.  You can also refer to the Jupyter Notebook primer on TPZ. 

In this task, you need to implement the following functions:

* normalize_filename()
* file2df()
* dirname2etf()
* filename2ticker()
* build_security_dfs()

It is recommended to complete the questions in chronological order, as later questions may depend on the successful implementation of previous questions.

More cells may be added to the notebook. If you do not want to include the cell in the output script that will be evaluated, please, tag the cell with `excluded_from_script`. You can display the tags for each cell in Jupyter Notebook: `View > Cell Toolbar > Tags`.

## Importing Libraries

In [1]:
# Importing the libraries for completing the task
# PLEASE DO NOT MODIFY THIS CELL

import numpy as np # linear algebra
import pandas as pd # data processing, CSV file I/O (e.g. pd.read_csv)
import os
import re

## About the Dataset

This dataset is originally sourced from Kaggle, and can be viewed [here](https://www.kaggle.com/borismarjanovic/price-volume-data-for-all-us-stocks-etfs). The dataset is structured as a directory that contains two subdirectories, named "ETFs" and "Stocks" respectively. Within each subdirectory are several thousand text files. The name of the each text file and the data in text file corresponds to the symbol of the security. For example ,the file "afk.us.txt" under the ETFs folder contains time series data for the security AFK. The contents of each file are various recorded metrics (opening price, closing price, the maximum price the stock traded at, the minimum, the volume traded, etc.) for the security on a particular date, presented in CSV Format.

For example, here are the first 5 lines of the file "afk.us.txt" under the ETFs subdirectory:

> Date,Open,High,Low,Close,Volume,OpenInt  
2008-07-14,34.598,34.664,34.598,34.664,4777,0  
2008-07-15,35.293,36.004,35,35.26,1670,0  
2008-07-16,34.672,34.672,33.827,33.827,1816,0  
2008-07-17,34.74,35.393,34.74,35.066,8294,0  
2008-07-18,34.749,34.749,33.493,34.33,9626,0  

## Resolving Data Inconsistencies

Later in this project, we will be combining the data attached to this notebook with some data fetched from the NasdaqTrader platform. To this end, the data attached to this notebook do not have symbols that match the conventions used by the Nasdaq Integrated Platform. Security ticker symbols (i.e. AAPL for Apple, GPRO for GoPro) often have suffixes attached to them to provide extra information about the security. For example, securities that are warrants (i.e. a security that when purchased gives the purchaser the right to purchase a company's stock at a fixed price on a later date) will be given the suffix "+" by the Nasdaq Integrated Platform (so the security ticker symbol would have the form "xxxx+"), but in the symbol conventions used by the filenames of this dataset, are denoted by the suffix "-ws" (so a warrant security would have a filename of the form "xxxx-ws.us.txt"). In addition, the filenames are all lowercase, whereas traditional ticker symbols are denoted with all uppercase letters. Resolving these inconsistencies is the purpose of the`normailze_filename` function. The function is partially implemented. Your task is to finish the implementation.

For more information on the NASDAQ Integrated Platform Suffix, feel free to explore [www.nasdaqtrader.com/trader.aspx?id=CQSsymbolconvention](https://www.nasdaqtrader.com/trader.aspx?id=CQSsymbolconvention).

In [2]:
def normalize_filename(filename):
    new_label = ''
    new_label = re.sub('-ws-', '+', filename)
    new_label = re.sub('-ws', '+', new_label)
    # === YOUR SOLUTION STARTS HERE ===
    # Do not change the code above.
    #
    # TODO 1: Following the examples above change the `new_label`
    #         variable so that "-cl" is replaced with "*" and
    #         "-u" is replaced with "=".
    
    # Do not change the code below.
    # === YOUR SOLUTION ENDS HERE ===
    if(re.fullmatch('.*-.*', new_label)):
        new_label = re.sub('-', '.', new_label)
    elif(re.fullmatch('.*_.*', new_label)):
        new_label = re.sub('_', '-', new_label)
    return new_label.upper()

## Deciding on Data Representation

When thinking about how to represent this data, our first instict might be to create one giant table of the following form, where the two last columns come from the filename and its placement in a specific directory:

| Date | Open | High | Low | Close | Volume | OpenInt | Ticker | Type |
|------|------|------|-----|-------|--------|---------|--------|------|
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | AAPL   | STOCK|
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | AAPL   | STOCK|
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | AAPL   | STOCK|
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | AAPL   | STOCK|
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | VOO    | ETF  |
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | VOO    | ETF  |
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | VOO    | ETF  |
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | ...    | ...  |

In the above table, all the information about each individual security's pricing on any given date is stored along with the information about the security itself in each record. However, notice that this creates a lot of unnecessary duplication in the "Ticker" and "Type" columns, and represents unnormalized data.

As mentioned in the MySQL primer, this creates vulnerabilities in our data representation to data input errors, inconsistencies, and potentially a lot of wasted space. Normalizing by creating a foreign key yields the following alternate structure:

Table 1:

| Date | Open | High | Low | Close | Volume | OpenInt | ID |       
|------|------|------|-----|-------|--------|---------|----|         
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | 1  |       
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | 1  |       
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | 1  |  
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | 1  |  
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | 2  |  
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | 2  |  
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | 2  |  
| ...  | ...  | ...  | ... | ...   | ...    |  ...    | ...|  

Table 2:

| ID | Ticker | Type |
|----|--------|------|
| 1  | AAPL   | STOCK|
| 2  | VOO    | ETF  |
|... | ...    | ...  |

Further, we can optimize the memory usage of Table 2 even more by changing the enumeration and meaning of the original "Type" column (new "ETF" column):

Table 2 Version 2:

| ID | Ticker | ETF  |
|----|--------|------|
| 1  | AAPL   | N    |
| 2  | VOO    | Y    |
|... | ...    | ...  |

Where "N" stands for "No" and "Y" stands for "Yes".

## Implementing Our Data Representation

Given that our dataset is quite fragmented i.e. consists of a large number of loose files, we could expect that our implementation of the above data representation will have to deal with this problem.

Let us first create the `file2df` procedure for generating 1 dataframe from 1 particular text file.


In [26]:
def file2df(dirname, filename, security_id):
    """ Given:
        dirname: a string representing the path to the directory containing the file with name filename
        filename: a string representing the name of a file in directory dirname
        securityID: a number representing the foreign key value assigned to this particular security
    
        This method when called with the correct arguments should return a pandas Dataframe containing the columns
        Date (of type datetime),Open, High, Low, Close, Volume, OpenInt, ID (of type int)
        populated with the data from the given file.
    """
    filepath = os.path.join(dirname, filename)
    df = pd.read_csv(filepath, sep=',')
    # === YOUR SOLUTION STARTS HERE ===
    # Do not change the code above.
    #
    # TODO 2: Add a column named ID that will have a constant
    #         value equal to the `securityID` argument.
    
    # TODO 3: Use pandas `to_datetime` function to parse the
    #         `Date` column to datetimes.
    
    # Do not change the code below.
    # === YOUR SOLUTION ENDS HERE ===
    return df

As we have already mentioned above an important information is encoded in a file's placement in a specific directory. Let us write a helper function `dirname2etf` to assist with the building of Table 2 (Version 2) as shown above.

In [27]:
def dirname2etf(dirname):
    """ Given the path to the directory containing the information of a particular security, 
    determine the contents of the "ETF" column of its record in Table 2
    """
    # === YOUR SOLUTION STARTS HERE ===
    # Do not change the code above.
    #
    # TODO 4: Implement the function that extracts the right most subdirectory
    #         of the provided `dirname` path. If the name of the subdirectory
    #         is "ETFs" then the function returns "Y" else it returns "N".
    
    # Do not change the code below.
    # === YOUR SOLUTION ENDS HERE ===

As we have already mentioned above an important information is encoded in a file's name. Let us write a helper function `filename2ticker` to assist with the building of Table 2 (Version 2) as shown above.

In [28]:
def filename2ticker(filename):
    """ Given the filename of the file containing the time series data on a particular security,
    determine the contents of the "Ticker" column of its record in Table 2
    """
    # === YOUR SOLUTION STARTS HERE ===
    # Do not change the code above.
    #
    # TODO 5: Implement the function that extracts the left most part of the
    #         provided filename (i.e., up till the first dot). The earlier
    #         implemented `normalize_filename` function is then used on the
    #         extracted part to create the desired result.
    
    # Do not change the code below.
    # === YOUR SOLUTION ENDS HERE ===

Finally, let's begin building the requisite python objects for our data representation:

In [29]:
def build_security_dfs():
    """ This method should return a tuple of the form (frame_time_series 
        frame_tickers) where frame_time_series is a pandas DataFrame representing
        the data found in the text files of the dataset along with the assigned
        foreign key ID, and frame_tickers is a DataFrame representing the second
        table of our chosen representation. 
        """
    directory = os.walk('./Data/')
    data_time_series = []
    ticker_inf = {'id': [], 'ticker': [], 'ETF': []}
    id_cntr = 0
    for dirname, _, filenames in sorted(directory):
        if len(filenames) == 0:
            continue
        type_ind = dirname2etf(dirname) #"Y" or "N"
        for filename in sorted(filenames):
            if os.path.getsize(os.path.join(dirname, filename)) > 0:
                # === YOUR SOLUTION STARTS HERE ===
                # Do not change the code above.
                #
                # TODO 6: Use `dirname`, `filename`, and `id_cntr` as
                #         arguments in the `file2df` that you implemented
                #         earlier to generate a dataframe object. Append
                #         the generated dataframe to the end of
                #         `data_time_series` list.
                
                # TODO 7: Use `filename` as the argument in the
                #         `filename2ticker` that you implemented earlier
                #         to generate a normalized ticker. Append the
                #         `id_cntr` to the list under the `id` key of the
                #         `ticker_inf` dictionary. Append the normalized
                #         ticker to the list under the `ticker` key of the
                #         `ticker_inf` dictionary. Append the `type_ind`
                #         to the list under the `ETF` key of the
                #         `ticker_inf` dictionary.
                
                # TODO 8: Increment the value of `id_ctr` variable by 1.
                pass
                
    
    frame_time_series = pd.concat(data_time_series)
    frame_tickers = pd.DataFrame(data=ticker_inf)
    return (frame_time_series, frame_tickers)

Run the following cell once after finishing all of the questions above. You will need the two generated CSV files as the input to the next task.

WARNING: The cell may run for quite some time (5-10 minutes). Just be patient and let it do its work.

In [7]:
# PLEASE DO NOT MODIFY THIS CELL

data_time_series, data_tickers = build_security_dfs()

data_time_series.to_csv('outputTimeSeries.csv', index=False)
data_tickers.to_csv('tickerInfo.csv', index=False)